# 00 · Prepare the NISP comparison cohort

Run this notebook from the project directory. It reads only English archives from the supplied NISP copy, excludes mismatched mother-tongue labels, and keeps the official test speakers untouched. The supplied archives contain a gzip stream inside another gzip stream. No original dataset files are modified.


In [1]:
from pathlib import Path
from collections import Counter
import gzip, os, re, shutil, tarfile
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.model_selection import StratifiedShuffleSplit

ROOT = Path.cwd()
SOURCE = Path(r'C:\Users\renny\Desktop\projects\speech proj\NISP-Dataset-master')
DATA = ROOT / 'data'
AUDIO = DATA / 'audio'
DATA.mkdir(exist_ok=True)
AUDIO.mkdir(exist_ok=True)
assert SOURCE.exists(), f'Missing dataset: {SOURCE}'
SEED = 42
LANGUAGES = {'Hin':'Hindi', 'Kan':'Kannada', 'Mal':'Malayalam', 'Tam':'Tamil', 'Tel':'Telugu'}
ALIASES = {'hindi':'Hindi', 'kannada':'Kannada', 'malayalam':'Malayalam',
           'malayam':'Malayalam', 'malyalam':'Malayalam', 'tamil':'Tamil', 'telugu':'Telugu'}


## 1. Audit speaker labels

The folder is a recorded-language grouping. For a native-language target, retain only speakers whose recorded mother tongue agrees with that grouping. The alias list corrects only unambiguous spelling or case differences.


In [2]:
columns = ['speaker_id','gender','mother_tongue','height','shoulder','waist',
           'weight','age','state','district']
speakers = pd.read_csv(SOURCE / 'total_spkrinfo.list', sep=r'\s+', names=columns, skiprows=1)
speakers['folder_label'] = speakers.speaker_id.str[:3].map(LANGUAGES)
speakers['normalized_l1'] = speakers.mother_tongue.str.lower().map(ALIASES)
speakers['label_match'] = speakers.normalized_l1.eq(speakers.folder_label)
speakers['gender'] = speakers.gender.str.lower()
assert speakers.speaker_id.is_unique and speakers.folder_label.notna().all()
assert set(speakers.gender) == {'male','female'}
speakers.to_csv(DATA / 'label_audit.csv', index=False)
print(f'{len(speakers)} metadata speakers; {speakers.label_match.sum()} match; '
      f'{(~speakers.label_match).sum()} excluded')
display(speakers.loc[~speakers.label_match, ['speaker_id','mother_tongue','folder_label']])


345 metadata speakers; 314 match; 31 excluded


,speaker_id,mother_tongue,folder_label
69,Hin_0070,Kannada,Hindi
71,Hin_0072,Malayalam,Hindi
74,Hin_0075,Kumauni,Hindi
80,Hin_0081,Odia,Hindi
82,Hin_0083,Gujarati,Hindi
89,Hin_0090,Bengali,Hindi
92,Hin_0093,Marathi,Hindi
96,Hin_0097,Telugu,Hindi
130,Tel_0028,Urdu,Telugu
157,Tel_0055,Urdu,Telugu


## 2. Make one speaker-disjoint split

The validation set is sampled only from official training speakers, stratified by language and gender. All utterances inherit their speaker's split.


In [3]:
def official_ids(filename):
    return {line.split('___')[0] for line in (SOURCE / filename).read_text().splitlines()
            if line.strip()}

train_ids = official_ids('train_spkrID')
test_ids = official_ids('test_spkrID')
assert not train_ids & test_ids
eligible = speakers.loc[speakers.label_match].copy()
assert set(eligible.speaker_id) <= train_ids | test_ids
eligible['split'] = np.where(eligible.speaker_id.isin(test_ids), 'test', 'train')
train_rows = eligible.loc[eligible.split.eq('train')]
strata = train_rows.folder_label + '_' + train_rows.gender
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
_, val_positions = next(splitter.split(train_rows, strata))
val_ids = set(train_rows.iloc[val_positions].speaker_id)
eligible.loc[eligible.speaker_id.isin(val_ids), 'split'] = 'val'
assert not (set(eligible.loc[eligible.split.eq('train'), 'speaker_id']) & val_ids)
assert set(eligible.loc[eligible.split.eq('test'), 'speaker_id']) <= test_ids
eligible.to_csv(DATA / 'speaker_split.csv', index=False)
display(pd.crosstab([eligible.split, eligible.folder_label], eligible.gender))


gender              female  male
split folder_label              
test  Hindi             10    26
      Kannada            9     9
      Malayalam         10    14
      Tamil              8    16
      Telugu             9    14
train Hindi             13    34
      Kannada            9    14
      Malayalam         12    17
      Tamil             11    17
      Telugu            10    14
val   Hindi              3     9
      Kannada            2     3
      Malayalam          3     4
      Tamil              3     5
      Telugu             3     3

## 3. Stream the double-gzip archives

Only matching English recordings are extracted. Files are written by speaker ID with checked basenames, so tar paths cannot escape the project directory. Re-running skips complete files.


In [4]:
class ConcatenatedParts:
    def __init__(self, paths):
        self.paths = iter(paths)
        self.current = None
    def read(self, n=-1):
        if n < 0:
            raise ValueError('Streaming reader requires a bounded read')
        output = bytearray()
        while len(output) < n:
            if self.current is None:
                try:
                    self.current = next(self.paths).open('rb')
                except StopIteration:
                    break
            block = self.current.read(n - len(output))
            if block:
                output.extend(block)
            else:
                self.current.close()
                self.current = None
        return bytes(output)
    def close(self):
        if self.current is not None:
            self.current.close()


In [5]:
WAV_NAME = re.compile(r'^(Hin|Kan|Mal|Tam|Tel)_(\d{4})_Eng_([mf])_(\d{4})\.wav$', re.I)
keep_ids = set(eligible.speaker_id)

def extract_english(prefix, label):
    archive_dir = SOURCE / f'{label}_master' / f'English_{label}'
    parts = sorted(archive_dir.glob('RECS.tar.gz.*'))
    if not parts:
        raise FileNotFoundError(archive_dir)
    source = ConcatenatedParts(parts)
    count = 0
    try:
        with gzip.GzipFile(fileobj=source) as outer:
            with tarfile.open(fileobj=outer, mode='r|gz') as archive:
                for member in archive:
                    if not member.isfile():
                        continue
                    basename = Path(member.name).name
                    match = WAV_NAME.fullmatch(basename)
                    if match is None or match.group(1).title() != prefix:
                        continue
                    speaker_id = '_'.join(basename.split('_')[:2])
                    if speaker_id not in keep_ids:
                        continue
                    destination = AUDIO / speaker_id / basename
                    if destination.exists() and destination.stat().st_size == member.size:
                        count += 1
                        continue
                    destination.parent.mkdir(exist_ok=True)
                    temporary = destination.with_suffix('.wav.part')
                    with archive.extractfile(member) as src, temporary.open('wb') as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                    if temporary.stat().st_size != member.size:
                        raise IOError(f'Incomplete extraction: {basename}')
                    os.replace(temporary, destination)
                    count += 1
    finally:
        source.close()
    print(f'{label}: {count} selected English WAV files')

for prefix, label in LANGUAGES.items():
    extract_english(prefix, label)


Hindi: 3697 selected English WAV files


Kannada: 2163 selected English WAV files


Malayalam: 2483 selected English WAV files


Tamil: 2419 selected English WAV files


Telugu: 2548 selected English WAV files


## 4. Check audio and transcripts

Transcript availability is audited but does not define the language label. The same valid audio manifest is used by both models.


In [6]:
def transcript_ids(speaker_id, label):
    scripts = SOURCE / f'{label}_master' / 'scripts'
    matches = list(scripts.rglob(f'{speaker_id}_Eng_*.txt'))
    if len(matches) != 1:
        return set()
    text = matches[0].read_text(encoding='utf-8', errors='replace')
    return {int(m.group(1)) for line in text.splitlines()
            if (m := re.match(r'^item(\d+)\s+', line))}

speaker_lookup = eligible.set_index('speaker_id')
texts = {sid: transcript_ids(sid, row.folder_label)
         for sid, row in speaker_lookup.iterrows()}
records, rejected = [], []
for path in sorted(AUDIO.glob('*/*.wav')):
    match = WAV_NAME.fullmatch(path.name)
    if match is None:
        rejected.append((str(path), 'filename'))
        continue
    sid = '_'.join(path.stem.split('_')[:2])
    if sid not in speaker_lookup.index:
        rejected.append((str(path), 'speaker'))
        continue
    row = speaker_lookup.loc[sid]
    if match.group(1).title() != sid[:3] or match.group(3).lower() != row.gender[0]:
        rejected.append((str(path), 'label_or_gender'))
        continue
    try:
        info = sf.info(path)
        if info.frames <= 0 or info.samplerate <= 0 or info.duration < 0.5:
            raise ValueError('empty or shorter than 0.5 seconds')
    except Exception as exc:
        rejected.append((str(path), f'audio: {exc}'))
        continue
    records.append({'utterance_id':path.stem, 'path':str(path.resolve()),
                    'speaker_id':sid, 'language':row.folder_label,
                    'gender':row.gender, 'split':row.split,
                    'duration_sec':info.duration, 'sample_rate':info.samplerate,
                    'channels':info.channels,
                    'has_transcript':int(match.group(4)) in texts[sid]})
manifest = pd.DataFrame(records).sort_values('utterance_id').reset_index(drop=True)
pd.DataFrame(rejected, columns=['path','reason']).to_csv(DATA / 'rejected_audio.csv', index=False)
assert not manifest.empty and manifest.utterance_id.is_unique
assert set(manifest.language) == set(LANGUAGES.values())
assert set(manifest.split) == {'train','val','test'}
manifest.to_csv(DATA / 'manifest.csv', index=False)
print(f'{len(manifest)} valid English utterances; {len(rejected)} rejected')
print('Transcript coverage:', f'{manifest.has_transcript.mean():.1%}')
display(pd.crosstab(manifest.split, manifest.language))


13310 valid English utterances; 0 rejected
Transcript coverage: 91.0%


language,Hindi,Kannada,Malayalam,Tamil,Telugu
split,,,,,
test,1385,867,1001,975,1107
train,1814,1065,1180,1113,1165
val,498,231,302,331,276


In [7]:
split_speakers = {split:set(rows.speaker_id)
                  for split, rows in manifest.groupby('split')}
assert all(not split_speakers[a] & split_speakers[b]
           for a in split_speakers for b in split_speakers if a != b)
assert split_speakers['test'] <= test_ids
print('Verified disjoint speakers:', {k:len(v) for k,v in split_speakers.items()})


Verified disjoint speakers: {'test': 125, 'train': 151, 'val': 38}
